# Stock factor evaluation

Evaluate whether a per-stock factor (one value per `(date, asset_id)`)
carries cross-sectional return predictability.


## Factor type

This recipe uses
`evaluate(metrics={"ic": ic()})` —
axes `(FactorScope.INDIVIDUAL, FactorDensity.DENSE)`.

Procedure: per-date Spearman ρ between factor and forward return,
aggregated to a Newey-West HAC t-statistic on `E[IC]`. PANEL mode
(`N ≥ 2`); `N = 1` raises `ModeAxisError` since there is no
cross-section to rank within.

Literature: [Grinold (1989)](../../reference/bibliography/);
[Newey & West (1987)](../../reference/bibliography/).


## Use this when

- Factor varies across assets at each date (per-stock signal, e.g.
  momentum, value, quality).
- Cross-section is wide (`N ≥ 30` for clean inference; `10 ≤ N < 30`
  emits `BORDERLINE_CROSS_SECTION_N`).
- Time series is at least 30 periods (`T < 20` is hard-blocked).

## What it tests

Null hypothesis `E[IC] = 0` — the factor has no rank-based
predictive ordering of forward returns across assets, on average
across dates. Standard error is NW HAC over the per-date IC series.

## Output to read

1. `profile.primary_p` — IC NW HAC p-value. For single-factor
   pre-registered analysis, compare against your nominal `α`
   directly; for N candidate factors, route through
   `multi_factor.bhy` to control FDR.
2. `profile.stats[StatCode.MEAN]` — sign + magnitude of average IC
   (cell identity is on `profile.config.metric`; the StatCode is
   intentionally cell-agnostic — see #187).
3. `profile.stats[StatCode.T_NW]` — the t-statistic that produced
   the p-value. Compare to ±2 as a rough sanity check.
4. `profile.warnings` — `UNRELIABLE_SE_SHORT_PERIODS` etc. flag
   data-quality risks that should change interpretation regardless
   of `primary_p`.

## 1. Setup


In [ ]:
from __future__ import annotations

import factrix as fx
from factrix.preprocess import compute_forward_return

print("factrix version:", fx.__version__)

## 2. Synthesise a cross-sectional panel

`make_cs_panel` produces a canonical panel with a target IC built
in. `ic_target=0.08` is a realistic effect size for a working
single-factor strategy.


In [ ]:
raw = fx.datasets.make_cs_panel(
    n_assets=100,
    n_dates=500,
    ic_target=0.08,
    seed=2024,
)
panel = compute_forward_return(raw, forward_periods=5)
print(f"panel shape={panel.shape}  N={panel['asset_id'].n_unique()}")

## 3. Evaluate

One factory call, one `evaluate()`. The factory commits to the
three axes; `evaluate()` derives `Mode` from the panel shape and
dispatches to the registered procedure.


In [ ]:
from factrix.metrics import ic_newey_west

results = fx.evaluate(
    panel,
    metrics={"ic": ic_newey_west()},
    factor_cols=["factor"],
    forward_periods=5,
)
profile = results["factor"]

print(f"primary_p    = {profile.metrics['ic'].p_value:.4g}")
print(f"structure    = {profile.cell[2]}")
print(f"ic_mean      = {profile.metrics['ic'].value:+.4f}")
print(f"ic_t_nw      = {profile.metrics['ic'].stat:+.2f}")

## 4. Inspect the full diagnose dict

`diagnose()` is the structured triage interface — same content the
individual-stat reads above derive from, in one dict for human
inspection or AI agent consumption.

In [ ]:
import json

print(json.dumps(profile.to_dict(), indent=2, default=str))

## 5. Sample-guard edge cases

This recipe runs the happy path. For the full `n_assets` × factory
behaviour matrix (small-N warnings, `N=1` fallbacks, `T<20` hard
block) see
[Guides § PANEL vs TIMESERIES](../../guides/panel-timeseries/).
Two notes for this cell specifically:

- `N < 30` emits `BORDERLINE_CROSS_SECTION_N` / `SMALL_CROSS_SECTION_N`.
- `N = 1` raises `ModeAxisError` with `suggested_fix=common_continuous(...)`
  — the factor would no longer be cross-sectional.


In [ ]:
print("stock_factor_evaluation: ok")